# MODULE G WORK_ORDER_v1 — mitdb/101 computation diary

**Executor:** Grok (xAI session 2026-09-17)  
**machine_sha:** `7ba7cd84b6d12c5b`  
**Branch:** `module-g-wo-v1-grok-fix1`  
**DOI baseline:** 10.5281/zenodo.22227622  
**Note:** single record, NOT cohort verdict.

This notebook is the reproducible diary of the end-to-end run on the first TIER-1 record (mitdb/101).
All cells are executable and produce the concrete numerical results of this leg.


## 0. Environment & seed

In [ ]:
import sys, platform, hashlib, json
from pathlib import Path
sys.path.insert(0, "scripts")
import numpy as np
from config import SEED
np.random.seed(SEED)

def machine_hash():
    raw = f"{platform.node()}-{platform.platform()}-{platform.machine()}"
    return hashlib.sha256(raw.encode()).hexdigest()[:16]

print("SEED =", SEED)
print("machine_sha =", machine_hash())
print("numpy", np.__version__)
print("platform", platform.platform())


## 1. Load & resample (§4.1) — first available ECG channel

In [ ]:
from data_io import load_and_resample
x, fs, meta = load_and_resample("mitdb", "101")
print(json.dumps(meta, indent=2))
print("signal shape:", x.shape, "dtype:", x.dtype)
print("duration_s:", round(meta["duration_s"], 1))


## 2. Takens embedding (§4.2) — τ from MI, m from FNN

In [ ]:
from embedding import mutual_information_lag, false_nearest_neighbors, delay_embed
tau, tau_s = mutual_information_lag(x, fs)
m, fnn_fracs = false_nearest_neighbors(x, tau)
print(f"m = {m} (M_MAX=6 → m_from_fnn_cap = {m == 6})")
print(f"τ = {tau} samples ({tau_s:.4f} s)")
print("FNN fractions:", [round(f, 4) for f in fnn_fracs])
emb = delay_embed(x, m, tau)
print("embedding shape:", emb.shape)


## 3. Windows + SG derivatives on embedded trajectory (§4.3)

In [ ]:
from embedding import make_windows, sg_smooth_and_derivatives
windows = make_windows(x, fs)[:30]   # this leg used max_windows=30
print("n_windows:", len(windows))
# first window demo
start, end = windows[0]
emb_seg = emb[start:min(end, len(emb))]
smoothed, vel, acc = sg_smooth_and_derivatives(emb_seg)
print("first window emb_seg shape:", emb_seg.shape)
print("vel shape (must be N'×m):", vel.shape)
print("acc shape:", acc.shape)


## 4. Per-window metrology (phase, θ, cond(g)) — summary of 30 windows

In [ ]:
from metrology import (compute_phase_and_bin, check_window_accepted, reeb_cycle,
                       theta_canon, theta_proj, theta_curv_TT, theta_curv_ASC,
                       anisotropy_proxy, bootstrap_cond_g)
from config import BOOTSTRAP_B

results = []
for wi, (start, end) in enumerate(windows):
    emb_seg = emb[start:min(end, len(emb))]
    if len(emb_seg) < 60:
        continue
    smoothed, vel, acc = sg_smooth_and_derivatives(emb_seg)
    phases, bin_idx, bin_stats = compute_phase_and_bin(emb_seg, vel)
    if not check_window_accepted(bin_stats):
        continue
    tc = theta_canon(vel, bin_stats, bin_idx)
    tp = theta_proj(vel, emb_seg, bin_stats, bin_idx)
    ttt = theta_curv_TT(vel, acc)
    tasc = theta_curv_ASC(vel, acc)
    aniso = anisotropy_proxy(vel)
    boot = bootstrap_cond_g(vel, B=BOOTSTRAP_B)
    results.append({
        "window_idx": wi,
        "theta_canon": float(tc), "theta_proj": float(tp),
        "theta_curv_TT": float(ttt), "theta_curv_ASC": float(tasc),
        "cond_g": float(aniso["cond_g"]), "log_cond_g": float(aniso["log_cond_g"]),
        "cond_g_bootstrap_median": float(boot["cond_g_bootstrap_median"]),
        "cond_g_ci95": [float(x) for x in boot["cond_g_ci95"]],
    })

print(f"accepted windows: {len(results)} / {len(windows)}")
print("first 3 windows (cond_g, theta_canon):")
for r in results[:3]:
    print(f"  w{r['window_idx']}: cond_g={r['cond_g']:.3f}, θ_canon={r['theta_canon']:.4f}")


## 5. Decision criteria §5 (E1–E4) — values from this run

In [ ]:
from scipy.stats import spearmanr
from config import E1_CI_HALFWIDTH, E4_WINDOW_FRACTION, E2_SPEARMAN, E3_SPEARMAN

n_accepted = len(results)
ci_half = []
for r in results:
    lo, hi = r["cond_g_ci95"]
    med = r["cond_g_bootstrap_median"]
    if np.isfinite(lo) and np.isfinite(hi) and med > 0:
        ci_half.append((hi - lo) / 2 / med)
    else:
        ci_half.append(np.nan)
ci_half = np.array(ci_half)
e1_frac = float(np.nanmean(ci_half < E1_CI_HALFWIDTH))
E1 = (e1_frac >= E4_WINDOW_FRACTION)

t_canon = np.array([r["theta_canon"] for r in results])
t_proj  = np.array([r["theta_proj"] for r in results])
t_tt    = np.array([r["theta_curv_TT"] for r in results])
t_asc   = np.array([r["theta_curv_ASC"] for r in results])
pairs = [(t_canon,t_proj),(t_canon,t_tt),(t_canon,t_asc),(t_proj,t_tt),(t_proj,t_asc),(t_tt,t_asc)]
rhos = []
for a,b in pairs:
    rho,_ = spearmanr(a,b) if len(a)>=3 else (0.0, None)
    rhos.append(float(rho) if np.isfinite(rho) else 0.0)
E2 = all(r >= E2_SPEARMAN for r in rhos)

log_cond = np.array([r["log_cond_g"] for r in results])
rho_e3,_ = spearmanr(t_canon, log_cond)
E3 = float(rho_e3) <= E3_SPEARMAN if np.isfinite(rho_e3) else False

# E4 placeholder (full isotropic n=100 done in companion artefact)
median_log_cond = float(np.nanmedian(log_cond))
E4_placeholder = median_log_cond > 1.0

print("E1 fraction (CI half-width < 0.5):", round(e1_frac, 3), "→ E1 =", E1)
print("Spearman matrix:", [round(r,4) for r in rhos], "→ E2 =", E2)
print("rho(θ_canon, log_cond_g) =", round(float(rho_e3),4), "→ E3 =", E3)
print("median log_cond_g =", round(median_log_cond,4), "→ E4_placeholder =", E4_placeholder)
print()
print("Final verdicts (this notebook re-execution):")
print({"E1": bool(E1), "E2": bool(E2), "E3": bool(E3), "E4": bool(E4_placeholder)})


## 6. Notes

- Blinding: annotations never loaded.
- Full isotropic null (c) n=100 on first 20 accepted windows is produced as a companion artefact and supersedes the E4 placeholder (see issue #3).
- This notebook freezes the diary of the primary metrics path.
